In [9]:
# SUPPORT IMPORTS
import os
import json
import numpy as np
from datasets import load_dataset

# RETRIEVAL IMPORTS
import faiss
from sentence_transformers import SentenceTransformer

# UTILITY IMPORTS
import torch

In [2]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

MODEL_NAME = "all-MiniLM-L6-v2"

INDEX_DIR = "Indices"
INDEX_FILE = f"{INDEX_DIR}/conll2023.faiss"
SENTENCE_STORE = f"{INDEX_DIR}/conll2023_sentences.json"

TOP_K = 3

In [3]:
print("LOADING EMBEDDER...")
EMBEDDER = SentenceTransformer(MODEL_NAME).to(DEVICE)
print("MODEL READY")

LOADING EMBEDDER...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


MODEL READY


In [4]:
print("LOADING DATASET...")
DATASET = load_dataset("conll2003", split="train")
SENTENCES = [" ".join(tokens) for tokens in DATASET["tokens"]]
print("TOTAL SENTENCES:", len(SENTENCES))

LOADING DATASET...


c:\Users\LENOVO\AppData\Local\Programs\Python\Python311\Lib\site-packages\datasets\load.py:1429: FutureWarning: The repository for conll2003 contains custom code which must be executed to correctly load the dataset. You can inspect the repository content at https://hf.co/datasets/conll2003
You can avoid this message in future by passing the argument `trust_remote_code=True`.
Passing `trust_remote_code=True` will be mandatory to load this dataset from the next major release of `datasets`.
  warnings.warn(


TOTAL SENTENCES: 14041


In [5]:
def EMBEDD_IT(TEXT_LIST):
    print("TURNING TEXT INTO VECTORS...")
    EMBEDDINGS = EMBEDDER.encode(
        TEXT_LIST,
        show_progress_bar=True,
        convert_to_numpy=True
    )

    return EMBEDDINGS

In [6]:
def MAKE_INDEX(EMBEDDINGS):

    DIM = EMBEDDINGS.shape[1]

    print("BUILDING FAISS INDEX...")

    INDEX = faiss.IndexFlatL2(DIM)

    INDEX.add(EMBEDDINGS.astype("float32"))

    print("INDEX SIZE:", INDEX.ntotal)

    return INDEX

In [7]:
def SAVE_STUFF(INDEX, SENTENCES):

    os.makedirs(INDEX_DIR, exist_ok=True)

    print("SAVING INDEX...")
    faiss.write_index(INDEX, INDEX_FILE)

    print("SAVING SENTENCE STORE...")
    with open(SENTENCE_STORE, "w") as f:
        json.dump(SENTENCES, f)

    print("DONE SAVING")

In [10]:
EMBEDDINGS = EMBEDD_IT(SENTENCES)

INDEX = MAKE_INDEX(EMBEDDINGS)

SAVE_STUFF(INDEX, SENTENCES)

TURNING TEXT INTO VECTORS...


Batches:   0%|          | 0/439 [00:00<?, ?it/s]

BUILDING FAISS INDEX...
INDEX SIZE: 14041
SAVING INDEX...
SAVING SENTENCE STORE...
DONE SAVING


In [11]:
def LOAD_STUFF():

    print("LOADING INDEX...")

    INDEX = faiss.read_index(INDEX_FILE)

    print("LOADING SENTENCES...")

    with open(SENTENCE_STORE) as f:
        SENTENCES = json.load(f)

    return INDEX, SENTENCES

In [12]:
def FIND_STUFF(QUERY, INDEX, SENTENCES, K=TOP_K):

    QUERY_VEC = EMBEDDER.encode([QUERY]).astype("float32")

    DIST, IDS = INDEX.search(QUERY_VEC, K)

    RESULTS = [SENTENCES[i] for i in IDS[0]]

    return RESULTS

In [13]:
INDEX, SENTENCES = LOAD_STUFF()

print(
    FIND_STUFF(
        "Who is the president of the company?",
        INDEX,
        SENTENCES
    )
)

LOADING INDEX...
LOADING SENTENCES...
['Inc .', 'Goldman , Sachs & Co .', "The startup company 's acting chairman is Jim Bidzos , the president of RSA Data Security , a unit of Security Dynamics Technologies Inc. as well as chairman of VeriSign ."]
